# COMP1013 Assignment 2.2 — Final Project
## Multimodal Product Type Classification with Amazon Berkeley Objects (ABO)

**Group**
- Juin Hong Tong — a1845206
- Syed Tousif Mahadi Farabi — a1977855
- Spoorthi — a1956386

This notebook implements the solution proposed in Project 2.1.

### Models compared
1. Text-only baseline
2. Image-only baseline
3. Feature-fusion multimodal model
4. Weighted late-fusion multimodal model

### Main evaluation
- Accuracy
- Macro-F1
- Weighted-F1
- Per-class F1
- Confusion matrices
- Feature-fusion training/validation curves
- Runtime / efficiency
- Failure cases

**Recommended:** Google Colab with a T4 GPU.


In [ ]:
%pip -q install datasets transformers scikit-learn pandas numpy matplotlib pillow requests joblib


In [ ]:
import os, io, json, time, random, math, zipfile, shutil, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import requests

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Final experiment settings.
N_CLASSES = 10
PER_CLASS = 250
TEST_SIZE = 0.15
VAL_SIZE = 0.15
IMAGE_BATCH = 32
TEXT_BATCH = 32
MLP_BATCH = 64
MLP_MAX_EPOCHS = 40
PATIENCE = 6

OUT = Path("/content/project22_results")
IMG_DIR = Path("/content/abo_project22_images")
OUT.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Final target size:", N_CLASSES, "classes x", PER_CLASS, "products/class")


## 1. Load metadata and repeat the data-quality checks

The final project does not assume the proposal-pilot statistics. It repeats the checks on the dataset.


In [ ]:
listings = load_dataset(
    "hyper3labs/amazon-berkeley-objects",
    "listings",
    split="train"
)

needed = ["item_id", "title", "product_type", "product_type_readable", "main_image_url"]
available = [c for c in needed if c in listings.column_names]
df = listings.select_columns(available).to_pandas()

quality = pd.DataFrame({
    "column": df.columns,
    "missing_n": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(100*df[c].isna().mean()) for c in df.columns],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
display(quality)
quality.to_csv(OUT/"data_quality.csv", index=False)

valid = df.dropna(subset=["title", "product_type", "main_image_url"]).copy()
valid = valid[
    valid["title"].astype(str).str.strip().ne("") &
    valid["product_type"].astype(str).str.strip().ne("") &
    valid["main_image_url"].astype(str).str.strip().ne("")
]

print("Metadata rows:", f"{len(df):,}")
print("Usable rows:", f"{len(valid):,}", f"({100*len(valid)/len(df):.1f}%)")
print("Usable product types:", valid["product_type"].nunique())


In [ ]:
type_counts = valid["product_type"].value_counts()

plt.figure(figsize=(10,5))
type_counts.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 product types before balancing")
plt.xlabel("Number of usable products")
plt.ylabel("")
plt.tight_layout()
plt.savefig(OUT/"01_original_class_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

display(type_counts.head(20).rename("count").to_frame())


## 2. Build the final balanced subset

The source data is strongly imbalanced. To compare classes and fusion methods fairly, the final experiment uses a balanced subset of high-frequency product types.

If image downloads fail, the notebook removes failed rows and then re-balances to the smallest remaining class.


In [ ]:
candidate_types = type_counts[type_counts >= PER_CLASS].head(N_CLASSES).index.tolist()
print("Selected classes:")
for c in candidate_types:
    print("-", c, "available:", int(type_counts[c]))

sampled = (
    valid[valid["product_type"].isin(candidate_types)]
    .groupby("product_type", group_keys=False)
    .apply(lambda x: x.sample(n=PER_CLASS, random_state=SEED))
    .reset_index(drop=True)
)

print("\nMetadata sample size:", len(sampled))


In [ ]:
def download_image(url, timeout=20):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        return None

records = []
t0 = time.time()

for i, row in sampled.iterrows():
    path = IMG_DIR / f"{i:05d}.jpg"
    if not path.exists():
        im = download_image(row["main_image_url"])
        if im is None:
            continue
        try:
            im.save(path, quality=90)
        except Exception:
            continue

    records.append({
        "path": str(path),
        "title": str(row["title"]),
        "label": str(row["product_type"]),
        "item_id": str(row["item_id"])
    })

download_seconds = time.time() - t0
downloaded = pd.DataFrame(records)

print(f"Downloaded/usable: {len(downloaded):,}/{len(sampled):,}")
print(f"Download success: {100*len(downloaded)/len(sampled):.1f}%")
print(f"Download time: {download_seconds:.1f}s")

counts = downloaded["label"].value_counts()
kept = counts[counts >= max(100, int(PER_CLASS*0.75))].index.tolist()
downloaded = downloaded[downloaded["label"].isin(kept)].copy()

min_count = int(downloaded["label"].value_counts().min())
balanced = (
    downloaded.groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=SEED))
    .reset_index(drop=True)
)

print("\nFinal balanced dataset:", balanced.shape)
display(balanced["label"].value_counts().rename("count").to_frame())
balanced.to_csv(OUT/"final_balanced_subset.csv", index=False)

plt.figure(figsize=(10,4.8))
balanced["label"].value_counts().sort_values().plot(kind="barh")
plt.title("Final balanced product-type subset")
plt.xlabel("Products")
plt.ylabel("")
plt.tight_layout()
plt.savefig(OUT/"02_final_class_distribution.png", dpi=220, bbox_inches="tight")
plt.show()


## 3. Stratified train / validation / test split

The test set is kept untouched until final evaluation. The validation set is used for MLP early stopping and late-fusion weight selection.


In [ ]:
train_temp, test_df = train_test_split(
    balanced,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=balanced["label"]
)

val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_temp,
    test_size=val_relative,
    random_state=SEED,
    stratify=train_temp["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

split_table = pd.DataFrame({
    "split": ["train","validation","test"],
    "n": [len(train_df),len(val_df),len(test_df)]
})
display(split_table)
split_table.to_csv(OUT/"split_sizes.csv", index=False)


## 4. Label encoding


In [ ]:
le = LabelEncoder()
le.fit(balanced["label"])
class_names = list(le.classes_)

for frame in (train_df, val_df, test_df):
    frame["y"] = le.transform(frame["label"])

print("Classes:", class_names)
joblib.dump(le, OUT/"label_encoder.joblib")


## 5. Image feature extraction — frozen pretrained ResNet-50

ResNet-50 is used as a pretrained image encoder. Its final classification layer is removed, leaving a 2048-dimensional image representation.


In [ ]:
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)
resnet.fc = nn.Identity()
resnet = resnet.to(DEVICE).eval()
image_transform = weights.transforms()

class ImagePathDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        im = Image.open(row["path"]).convert("RGB")
        return image_transform(im), int(row["y"])

@torch.no_grad()
def extract_image_features(frame, batch_size=IMAGE_BATCH):
    ds = ImagePathDataset(frame)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    feats, ys = [], []
    start = time.time()
    for xb, yb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        fb = resnet(xb).detach().cpu().numpy()
        feats.append(fb)
        ys.append(yb.numpy())
    return np.vstack(feats), np.concatenate(ys), time.time()-start

Xtr_img, ytr, img_t_train = extract_image_features(train_df)
Xva_img, yva, img_t_val = extract_image_features(val_df)
Xte_img, yte, img_t_test = extract_image_features(test_df)

print("Image features:", Xtr_img.shape)
print("Image feature extraction seconds:", round(img_t_train+img_t_val+img_t_test,2))


## 6. Text feature extraction — frozen DistilBERT

Titles are tokenised with DistilBERT. The final hidden states are mean-pooled over non-padding tokens to produce one text vector per product.


In [ ]:
TEXT_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL).to(DEVICE).eval()

@torch.no_grad()
def extract_text_features(frame, batch_size=TEXT_BATCH, max_length=64):
    texts = frame["title"].astype(str).tolist()
    feats = []
    start = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tok = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        out = text_encoder(**tok).last_hidden_state
        mask = tok["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        feats.append(pooled.detach().cpu().numpy())

    return np.vstack(feats), time.time()-start

Xtr_txt, txt_t_train = extract_text_features(train_df)
Xva_txt, txt_t_val = extract_text_features(val_df)
Xte_txt, txt_t_test = extract_text_features(test_df)

print("Text features:", Xtr_txt.shape)
print("Text feature extraction seconds:", round(txt_t_train+txt_t_val+txt_t_test,2))


## 7. Standardise embeddings

The scalers are fit on training data only.


In [ ]:
img_scaler = StandardScaler()
txt_scaler = StandardScaler()

Xtr_img_s = img_scaler.fit_transform(Xtr_img).astype("float32")
Xva_img_s = img_scaler.transform(Xva_img).astype("float32")
Xte_img_s = img_scaler.transform(Xte_img).astype("float32")

Xtr_txt_s = txt_scaler.fit_transform(Xtr_txt).astype("float32")
Xva_txt_s = txt_scaler.transform(Xva_txt).astype("float32")
Xte_txt_s = txt_scaler.transform(Xte_txt).astype("float32")

joblib.dump(img_scaler, OUT/"image_scaler.joblib")
joblib.dump(txt_scaler, OUT/"text_scaler.joblib")


## 8. Unimodal baselines

The same labels and test set are used for both baselines.


In [ ]:
def fit_logreg(X, y):
    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="auto",
        random_state=SEED
    )
    start = time.time()
    model.fit(X, y)
    return model, time.time()-start

img_clf, img_train_seconds = fit_logreg(Xtr_img_s, ytr)
txt_clf, txt_train_seconds = fit_logreg(Xtr_txt_s, ytr)

joblib.dump(img_clf, OUT/"image_logreg.joblib")
joblib.dump(txt_clf, OUT/"text_logreg.joblib")

print("Image baseline train seconds:", round(img_train_seconds,2))
print("Text baseline train seconds:", round(txt_train_seconds,2))


## 9. Feature-fusion model — concatenated ResNet + DistilBERT embeddings

This is the first formal multimodal solution from the proposal. Image and text embeddings are concatenated and passed through a small MLP.


In [ ]:
Xtr_fusion = np.hstack([Xtr_img_s, Xtr_txt_s]).astype("float32")
Xva_fusion = np.hstack([Xva_img_s, Xva_txt_s]).astype("float32")
Xte_fusion = np.hstack([Xte_img_s, Xte_txt_s]).astype("float32")

class FusionMLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        return self.net(x)

def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=MLP_BATCH, shuffle=shuffle)

train_loader = make_loader(Xtr_fusion, ytr, True)
val_loader = make_loader(Xva_fusion, yva, False)

fusion_model = FusionMLP(Xtr_fusion.shape[1], len(class_names)).to(DEVICE)
opt = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

history = {"train_loss":[], "val_loss":[], "val_macro_f1":[]}
best_state = None
best_f1 = -1
wait = 0
start_train = time.time()

for epoch in range(1, MLP_MAX_EPOCHS+1):
    fusion_model.train()
    running = 0.0
    n_seen = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = fusion_model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        running += float(loss.item()) * len(xb)
        n_seen += len(xb)
    train_loss = running / n_seen

    fusion_model.eval()
    val_losses, preds, truth = [], [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = fusion_model(xb)
            val_losses.append(float(loss_fn(logits, yb).item()) * len(xb))
            preds.extend(logits.argmax(1).cpu().numpy())
            truth.extend(yb.cpu().numpy())

    val_loss = sum(val_losses)/len(truth)
    val_f1 = f1_score(truth, preds, average="macro")
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(val_f1)

    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val macro-F1 {val_f1:.4f}")

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        best_state = {k:v.detach().cpu().clone() for k,v in fusion_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

fusion_train_seconds = time.time() - start_train
fusion_model.load_state_dict(best_state)
fusion_model = fusion_model.to(DEVICE).eval()
torch.save(fusion_model.state_dict(), OUT/"feature_fusion_mlp.pt")

print("Best validation macro-F1:", round(best_f1,4))
print("Fusion training seconds:", round(fusion_train_seconds,2))


In [ ]:
epochs = np.arange(1, len(history["train_loss"])+1)

plt.figure(figsize=(8,4.5))
plt.plot(epochs, history["train_loss"], marker="o", label="Train loss")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Feature-fusion MLP: training vs validation loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"03_feature_fusion_loss_curve.png", dpi=220, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,4.5))
plt.plot(epochs, history["val_macro_f1"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")
plt.title("Feature-fusion MLP validation performance")
plt.tight_layout()
plt.savefig(OUT/"04_feature_fusion_val_f1.png", dpi=220, bbox_inches="tight")
plt.show()


## 10. Late fusion — validation-tuned weighting

This is the second formal multimodal solution. Separate image and text classifiers produce class probabilities, which are combined with a weight selected only on validation data.


In [ ]:
img_val_prob = img_clf.predict_proba(Xva_img_s)
txt_val_prob = txt_clf.predict_proba(Xva_txt_s)

weight_rows = []
for alpha in np.arange(0.0, 1.01, 0.1):
    fused = alpha*img_val_prob + (1-alpha)*txt_val_prob
    pred = fused.argmax(1)
    score = f1_score(yva, pred, average="macro")
    weight_rows.append({"image_weight":float(alpha), "text_weight":float(1-alpha), "val_macro_f1":float(score)})

weight_table = pd.DataFrame(weight_rows)
display(weight_table)
weight_table.to_csv(OUT/"late_fusion_weight_search.csv", index=False)

best_alpha = float(weight_table.loc[weight_table["val_macro_f1"].idxmax(), "image_weight"])
print("Best image weight:", best_alpha)
print("Best text weight:", 1-best_alpha)


## 11. Final test evaluation

The untouched test set is used once for the final comparison.


In [ ]:
def fusion_predict(model, X, batch_size=256):
    model.eval()
    preds = []
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size]).to(DEVICE)
            logits = model(xb)
            probs.append(torch.softmax(logits,1).cpu().numpy())
            preds.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(preds), np.vstack(probs)

# Baselines
img_pred = img_clf.predict(Xte_img_s)
txt_pred = txt_clf.predict(Xte_txt_s)
img_prob = img_clf.predict_proba(Xte_img_s)
txt_prob = txt_clf.predict_proba(Xte_txt_s)

# Feature fusion
feature_pred, feature_prob = fusion_predict(fusion_model, Xte_fusion)

# Late fusion
late_prob = best_alpha*img_prob + (1-best_alpha)*txt_prob
late_pred = late_prob.argmax(1)

def metric_row(name, pred):
    return {
        "model": name,
        "accuracy": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro")),
        "weighted_f1": float(f1_score(yte, pred, average="weighted"))
    }

results = pd.DataFrame([
    metric_row("Text only", txt_pred),
    metric_row("Image only", img_pred),
    metric_row("Feature fusion", feature_pred),
    metric_row("Late fusion", late_pred),
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(results.style.format({"accuracy":"{:.3f}","macro_f1":"{:.3f}","weighted_f1":"{:.3f}"}))
results.to_csv(OUT/"final_model_results.csv", index=False)


In [ ]:
plt.figure(figsize=(8.5,4.8))
x = np.arange(len(results))
w = 0.27
plt.bar(x-w, results["accuracy"], width=w, label="Accuracy")
plt.bar(x, results["macro_f1"], width=w, label="Macro-F1")
plt.bar(x+w, results["weighted_f1"], width=w, label="Weighted-F1")
plt.xticks(x, results["model"], rotation=10)
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("Final model comparison on the test set")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"05_final_model_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


## 12. Per-class F1 and confusion matrices


In [ ]:
pred_map = {
    "text_only": txt_pred,
    "image_only": img_pred,
    "feature_fusion": feature_pred,
    "late_fusion": late_pred
}

per_class_frames = []
for model_name, pred in pred_map.items():
    rep = classification_report(yte, pred, target_names=class_names, output_dict=True, zero_division=0)
    pc = pd.DataFrame(rep).T.loc[class_names, ["precision","recall","f1-score","support"]].reset_index()
    pc = pc.rename(columns={"index":"class"})
    pc["model"] = model_name
    per_class_frames.append(pc)

per_class = pd.concat(per_class_frames, ignore_index=True)
per_class.to_csv(OUT/"per_class_metrics.csv", index=False)

display(
    per_class[per_class["model"].isin(["feature_fusion","late_fusion"])]
    .pivot(index="class", columns="model", values="f1-score")
    .round(3)
)


In [ ]:
for model_name in ["feature_fusion", "late_fusion"]:
    pred = pred_map[model_name]
    cm = confusion_matrix(yte, pred)
    fig, ax = plt.subplots(figsize=(10,9))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"Confusion matrix — {model_name.replace('_',' ').title()}")
    plt.tight_layout()
    plt.savefig(OUT/f"06_confusion_matrix_{model_name}.png", dpi=220, bbox_inches="tight")
    plt.show()


## 13. Failure-case analysis

The table below focuses on cases where the best multimodal model is wrong. This is useful for the report discussion and Q&A.


In [ ]:
best_model_name = results.iloc[0]["model"]
if best_model_name == "Feature fusion":
    best_pred = feature_pred
    best_prob = feature_prob
elif best_model_name == "Late fusion":
    best_pred = late_pred
    best_prob = late_prob
elif best_model_name == "Image only":
    best_pred = img_pred
    best_prob = img_prob
else:
    best_pred = txt_pred
    best_prob = txt_prob

failure = test_df.reset_index(drop=True).copy()
failure["true_label"] = le.inverse_transform(yte)
failure["pred_label"] = le.inverse_transform(best_pred)
failure["confidence"] = best_prob.max(axis=1)
failure["correct"] = failure["true_label"] == failure["pred_label"]

failure_cases = (
    failure[~failure["correct"]]
    .sort_values("confidence", ascending=False)
    [["item_id","title","true_label","pred_label","confidence","path"]]
    .head(30)
)

display(failure_cases)
failure_cases.to_csv(OUT/"failure_cases.csv", index=False)


## 14. Efficiency and overfitting summary


In [ ]:
timing = pd.DataFrame([
    {"stage":"Image embedding extraction", "seconds":img_t_train+img_t_val+img_t_test},
    {"stage":"Text embedding extraction", "seconds":txt_t_train+txt_t_val+txt_t_test},
    {"stage":"Image baseline training", "seconds":img_train_seconds},
    {"stage":"Text baseline training", "seconds":txt_train_seconds},
    {"stage":"Feature-fusion MLP training", "seconds":fusion_train_seconds},
])
display(timing.round(2))
timing.to_csv(OUT/"timing_summary.csv", index=False)

# Simple overfitting indicators for report discussion.
train_feature_pred, _ = fusion_predict(fusion_model, Xtr_fusion)
train_feature_f1 = f1_score(ytr, train_feature_pred, average="macro")
test_feature_f1 = f1_score(yte, feature_pred, average="macro")

overfit = {
    "feature_fusion_train_macro_f1": float(train_feature_f1),
    "feature_fusion_test_macro_f1": float(test_feature_f1),
    "train_test_gap": float(train_feature_f1-test_feature_f1),
    "best_validation_macro_f1": float(best_f1)
}
print(overfit)


## 15. Ethics, bias and deployment notes

These points should be discussed in the report:

- **Source/product bias:** ABO reflects Amazon catalogue content rather than all products in the world.
- **Language bias:** this project filters to English titles.
- **Class-selection bias:** only high-frequency product types are used in the final balanced experiment.
- **Brand leakage:** product titles can contain brand/model tokens that make some classes easier.
- **Image quality:** studio catalogue images may not represent user photographs or deployment images.
- **No sensitive-person inference:** the task predicts product type, not demographic or personal attributes.
- **Mitigation:** report class-level metrics, keep the test split stratified, disclose the balancing strategy, analyse failure cases, and avoid claims outside the selected classes.


## 16. Save the final reproducibility summary and package all outputs


In [ ]:
summary = {
    "seed": SEED,
    "device": DEVICE,
    "metadata_rows": int(len(df)),
    "usable_rows": int(len(valid)),
    "usable_percentage": float(100*len(valid)/len(df)),
    "usable_product_types": int(valid["product_type"].nunique()),
    "selected_classes": class_names,
    "final_balanced_n": int(len(balanced)),
    "train_n": int(len(train_df)),
    "val_n": int(len(val_df)),
    "test_n": int(len(test_df)),
    "best_late_fusion_image_weight": best_alpha,
    "best_late_fusion_text_weight": 1-best_alpha,
    "results": results.to_dict(orient="records"),
    "overfitting": overfit,
    "timing_seconds": timing.set_index("stage")["seconds"].to_dict(),
    "software": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available())
    }
}

with open(OUT/"results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:
# Create one downloadable ZIP with all result tables and figures.
zip_path = "/content/COMP1013_Project22_RESULTS.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Created:", zip_path)
print("\n=== FINAL RESULTS TO SEND FOR REPORT CHECKING ===")
display(results)
print("\nBest late-fusion weights: image =", best_alpha, ", text =", 1-best_alpha)
print("Feature-fusion train Macro-F1:", round(train_feature_f1,4))
print("Feature-fusion test Macro-F1:", round(test_feature_f1,4))
print("Results ZIP:", zip_path)


In [ ]:
# Optional: trigger a browser download in Google Colab.
try:
    from google.colab import files
    files.download("/content/COMP1013_Project22_RESULTS.zip")
except Exception as e:
    print("Automatic download is only available in Google Colab.")
    print("You can still download /content/COMP1013_Project22_RESULTS.zip from the Files panel.")
